In [ ]:
from recon.data import fetch_all_tutorial_data
fetch_all_tutorial_data(data_dir='./data')

  0%|                                              | 0.00/24.3M [00:00<?, ?B/s]

Downloaded to: /home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/perturbation_tuto/rna.h5ad


  0%|                                              | 0.00/1.79G [00:00<?, ?B/s]

Downloaded to: /home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/perturbation_tuto/rna_treated.h5ad


0.00B [00:00, ?B/s]

Downloaded to: /home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/perturbation_tuto/grn.csv


  0%|                                               | 0.00/784M [00:00<?, ?B/s]

Downloaded to: /home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/build_grn_tuto/pbmc10x.h5mu


{'perturbation_tuto/rna.h5ad': '/home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/perturbation_tuto/rna.h5ad',
 'perturbation_tuto/rna_treated.h5ad': '/home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/perturbation_tuto/rna_treated.h5ad',
 'perturbation_tuto/grn.csv': '/home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/perturbation_tuto/grn.csv',
 'build_grn_tuto/pbmc10x.h5mu': '/home/marcelo.hurtado/Documents/CellTBool/Tutorials/data/build_grn_tuto/pbmc10x.h5mu'}

## Predicting Treatment Effects with ReCoN

This tutorial demonstrates how to use ReCoN to predict the cell-type-specific effects of a molecular treatment. You will learn how to:

1. Build a multilayer network integrating GRNs and cell-cell communication
2. Select a molecule (ligand) to perturb
3. Predict direct effects (on receptor-expressing cells) and indirect effects (via cell-cell signaling)
4. Compare predicted responses across cell types

Use case: Given a therapeutic molecule (e.g., a ligand or antibody), predict which genes will be affected in each cell type and identify cell-type-specific responses.

In [ ]:
import numpy as np
import scanpy as sc  # single cell data
import pandas as pd  # data manipulation
import liana as li  # cell communication
import recon  # multilayer and perturbation prediction
import recon.data

Load data

In [ ]:
rna = sc.read_h5ad("./data/perturbation_tuto/rna.h5ad", backed="r")
rna = rna[:, :2000].to_memory()

Let’s check what cell types are present in this dataset

In [ ]:
rna.obs["celltype"].unique().tolist()[:5]

['B_cell', 'ILC', 'Macrophage', 'MigDC', 'Monocyte']

## Build the Multilayer Network

ReCoN integrates three main components:

1. Gene Regulatory Networks (GRNs) - TF → target gene relationships
2. Cell-Cell Communication (CCC) - Ligand-receptor interactions between cell types
3. Receptor-Gene Links - How receptors connect to intracellular signaling

###  1. Import Gene Regulatory Network

You can either generate GRNs directly with ReCoN or import a previously generated one.

In [ ]:
grn_path = "./data/perturbation_tuto/grn.csv"
grn = pd.read_csv(grn_path)
grn = grn.sort_values(by="weight", ascending=False)[:500_000]
grn["source"] = grn["source"].str.capitalize()
grn["source"] = grn["source"] + '_TF'
grn["target"] = grn["target"].str.capitalize()
grn.head(3)

,Unnamed: 0,target,source,weight
0,0,Pax5,Mbd1_TF,0.000095
2,2,Pax5,Smad5_TF,0.000092
1,1,Pax5,Smad1_TF,0.000092


### 2. Compute Cell-Cell Communication

The cell-cell communication is inferred through LIANA+, an external package dedicated to this task.

In [ ]:
li.method.cellphonedb(rna, 
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name="mouseconsensus",
            expr_prop=0.00,
            use_raw=False,
            groupby="celltype",
            verbose=True, key_added='cpdb_res')

Using resource `mouseconsensus`.
Using `.X`!
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
901 features of mat are empty, they will be removed.
Make sure that normalized counts are passed!
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.94 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 1296 samples and 15 features


100%|██████████| 1000/1000 [00:02<00:00, 462.55it/s]


In [ ]:
ccc_network = rna.uns["cpdb_res"].copy()
ccc_network = ccc_network[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_network = ccc_network.rename(columns={
    "lr_means": "weight",
    "source": "celltype_source",
    "target": "celltype_target",
    "ligand": "source",
    "receptor": "target"
})
ccc_network = ccc_network[ccc_network['weight'] != 0]

In [ ]:
ccc_network.head(3)

,source,target,weight,celltype_source,celltype_target
1457,Mrc1,Ptprc,3.505000,Macrophage,cDC2
1537,Mrc1,Ptprc,3.455000,cDC2,cDC2
1473,Mrc1,Ptprc,3.327027,Monocyte,cDC2


### 3. Load Receptor-Gene Links

These links connect membrane receptors to downstream target genes in the GRN, enabling signal propagation from extracellular to intracellular networks.

In [ ]:
receptor_genes = recon.data.load_receptor_genes("mouse_receptor_gene_from_NichenetPKN")
# for human, use "human_receptor_gene_from_NichenetPKN"

genes = np.unique(grn['source'].tolist() + grn['target'].tolist())
receptor_genes = receptor_genes[receptor_genes['target'].isin(genes)]
receptor_genes.head()

,source,target,weight
2,A1bg,Abca1,0.005156
3,A1bg,Abcb1a,0.005877
4,A1bg,Abcb1b,0.005877
7,A1bg,Acsl1,0.005915
8,A1bg,Adk,0.005092


## Select a Molecule to Perturb

For treatment effect prediction, we need to choose a molecule (ligand) to simulate as a treatment. There are several strategies:

- Strategy 1: Choose a receptor with high variance across cell types, then find its ligands.

This helps identify molecules that may have cell-type-specific effects:

In [ ]:
# variance
ccc_network.groupby("target")["weight"].var().sort_values(ascending=False).head(3)

target
Ptprc    0.424900
Sell     0.078597
Cd244    0.061958
Name: weight, dtype: float32

Now identify the natural ligands of this receptor. You can either:

- Use a designed molecule targeting this receptor
- Choose one of its natural ligands from the cell communication network

In [ ]:
ligands = ccc_network[ccc_network["target"]=="Cd244"]['source'].unique().tolist()
ligands

['Cd48']

## Run Treatment Effect Prediction

The multicell_targets function builds the multilayer network and runs Random Walk with Restart to predict treatment effects.

Direct vs Indirect Effects

- Direct effect: Impact on cells that directly express the receptor for your molecule
- Indirect effect: Impact propagated through cell-cell communication from directly affected cells to other cell types

In [ ]:
print(grn.shape)
print(receptor_genes.shape)
print(ccc_network.shape)

(500000, 4)
(434183, 3)
(465, 5)


In [ ]:
%%time
direct_effect, indirect_effect = recon.explore.multicell_targets(
        seeds=["Cd48"],
        celltypes=["B_cell", "pDC", "Macrophage", "NK_cell", "T_cell_CD4", "T_cell_CD8"],
        grn=grn,
        receptor_grn=receptor_genes,
        ccc=ccc_network,
        grn_graph_weighted=True,
        receptor_grn_graph_weighted=True,
        receptor_graph_weighted=False,
        cell_communication_graph_weighted=True,
        cell_communication_graph_directed=False,
        restart_proba=0.6,
        extend_seeds=True,
        njobs=1
    )

Processing celltype 1/6: B_cell


/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:122: UserWarning: 
                No receptor_graph provided,
                an empty receptor graph will be created.
                


Processing celltype 2/6: pDC
Processing celltype 3/6: Macrophage
Processing celltype 4/6: NK_cell
Processing celltype 5/6: T_cell_CD4
Processing celltype 6/6: T_cell_CD8


/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:387: UserWarning: The celltypes dictionary was converted toa list of Celltype objects.
The keys of the dictionary will be the celltype names.


Computing intracellular contributions and direct effect...
Computing intercellular contributions and indirect effect...


...

Seeds are provided as a dictionary with weights per seed.
Creating a multixrank object with seeds as a dictionary.
Identifying produced ligands in response to the perturbation.


/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:265: RuntimeWarning: invalid value encountered in divide
 83%|████████▎ | 5/6 [08:09<01:38, 98.09s/it]

Seeds are provided as a dictionary with weights per seed.
Creating a multixrank object with seeds as a dictionary.
Identifying produced ligands in response to the perturbation.


/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:265: RuntimeWarning: invalid value encountered in divide
100%|██████████| 6/6 [09:47<00:00, 97.84s/it]
[Parallel(n_jobs=1)]: Done   6 out of   6 | elapsed:  9.8min finished


CPU times: user 11min 1s, sys: 14.4 s, total: 11min 15s
Wall time: 11min 14s


## Analyze Treatment Effects

View Direct and Indirect Effects

Direct effects: Genes affected in cells expressing the target receptor

In [ ]:
direct_effect.head()

celltype_target,B_cell,pDC,Macrophage,NK_cell,T_cell_CD4,T_cell_CD8
gene,,,,,,
Zzz3,0.0,0.0,0.0,0.0,0.0,0.0
Zzef1,0.0,0.0,0.0,0.0,0.0,0.0
Zyx,0.0,0.0,0.0,0.0,0.0,0.0
Zyg11b,0.0,0.0,0.0,0.0,0.0,0.0
Zxdc,0.0,0.0,0.0,0.0,0.0,0.0


Indirect effects: Genes affected through cell-cell communication cascades

In [ ]:
indirect_effect.head()

Combine Direct and Indirect Effects

In [ ]:
total_effect = recon.explore.combine_effects(direct_effect, indirect_effect, alpha=0.8)
total_effect.head()

Plot effects

In [ ]:
%matplotlib inline
total_effect.plot.scatter(x='B_cell', y='pDC')

Visualize Cell Type Correlations

Scatter plots showing correlation of predicted effects between cell types:

In [ ]:
%matplotlib inline
total_effect.plot.scatter(x='B_cell', y='Macrophage')

In [ ]:
%matplotlib inline
total_effect.plot.scatter(x='pDC', y='Macrophage')

## Compare Cell-Type-Specific Responses

A key application of ReCoN is identifying differential responses between cell types. This helps understand:

- Which cell types are most affected by a treatment
- Which genes drive cell-type-specific responses
- Potential off-target effects on non-target cell populations

T cells CD4+ vs B cells

Compare predicted treatment response between T helper cells and B cells:

In [ ]:
(total_effect["T_cell_CD4"] - total_effect["B_cell"]).sort_values(ascending=False)[:20]

Visualize with a comparison plot highlighting the most differential genes:

In [ ]:
recon.plot.plot_celltype_comparison(total_effect, "T_cell_CD4", "B_cell", quantile=0.998)

B cells vs T cells CD4+

Now look at genes with higher scores in B cells:

In [ ]:
(total_effect["B_cell"] - total_effect["T_cell_CD4"]).sort_values(ascending=False)[:20]

In [ ]:
recon.plot.plot_celltype_comparison(total_effect, "B_cell", "T_cell_CD4", quantile=0.999)

Macrophages vs Other Cell Types

Compare macrophage response to the average of all other cell types:

In [ ]:
(total_effect["Macrophage"] - total_effect.loc[:, ~total_effect.columns.isin(["Macrophage"])].mean(1)).sort_values(ascending=False)[:20]

In [ ]:
total_effect["average_no_macrophage"] = total_effect.loc[:, ~total_effect.columns.isin(["Macrophage"])].mean(axis=1)
recon.plot.plot_celltype_comparison(total_effect, "Macrophage", "average_no_macrophage", quantile=0.998)
# delete to not include it by mistake later
del total_effect["average_no_macrophage"]

## Gene Ranking Analysis

### Compare Gene Ranks Across Cell Types

We can also compare the rank of each gene across different cell types to identify genes that are consistently high or show cell-type-specific rankings:

In [ ]:
total_effect.rank().sort_values(by="NK_cell")[:10]

### Focus on Known Cell Type Markers

Let’s check how known immune cell markers rank in each cell type:

In [ ]:
markers = pd.Series([
    "Ptprc",   # pan-leukocyte (CD45)
    "Cd3e", "Trac",              # T cells
    "Cd4", "Cd8a",               # helper vs cytotoxic T
    "Ncr1", "Klrk1",             # NK cells
    "Ms4a1", "Cd19", "Cd79a",    # B cells
    "Sdc1",                      # plasmablasts/plasma cells
    "Lyz2", "Adgre1", "Csf1r",   # monocytes/macrophages
    "Itgax", "Zbtb46",           # conventional dendritic cells
    "Siglech",                   # plasmacytoid dendritic cells
    "Ly6g", "S100a8", "S100a9"   # neutrophils
])

markers = markers[markers.isin(total_effect.index).values]

total_effect.rank(ascending=False).loc[markers].sort_values(by="NK_cell")

Summary

In this tutorial, you learned how to:

1. Build a multilayer network integrating GRNs, cell-cell communication, and receptor-gene links
2. Select a molecule to simulate as a treatment based on receptor variance
3. Predict treatment effects using multicell_targets() to get direct and indirect effects
4. Combine effects with the α parameter (default 0.8 for direct-weighted)
5. Compare cell-type responses to identify differential effects and cell-type-specific genes